# Notebook 02 — Action-data preparation

This stage uses the actual Notebook 01 inventory and project case examples. It prepares an annotation workflow and a trainable baseline without converting personas into student-performance labels. The Python module is shared with the command-line script. No trained performance or learning improvement is claimed.

The sequence is: source eligibility → conversation parsing → source spans → human review → grouped splits → held-out evaluation. General dialogue acts and rubric-linked student behavior are separate tasks.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'PRD.md').exists())
sys.path.insert(0, str(ROOT / 'backend'))
from backend_kelsch.scripts.prepare_action_data import build_report
from backend_kelsch.app.action_data.preparation import parse_dialogue, validate_annotations, grouped_split, evaluate
report = build_report(ROOT)
print('Prior gate:', report['gates']['prior_structural_gate'])
print('Model gate:', report['gate'])

## 1. What the source check found

The Hugging Face API is accessible. The recorded revision and file inventory below come from that API. No license file or dataset card establishes reuse permission or synthetic provenance. Repository context explicitly prohibits using unlicensed or real patient data. Consequently no upstream dialogue contents were downloaded. The upstream format remains unverified; the parser demonstration below is a synthetic mechanics example.

In [ ]:
display(pd.Series(report['external_source']))

## 2. Actual supplied examples and provenance

These are patient persona lines from the three project cases. They are isolated authored speech, not recorded encounters. They can enter an annotation queue but cannot serve as student-skill labels. Compare their source pointer and hash with the original case JSON.

In [ ]:
examples = [dict(case=c['conversation_id'], speaker=t['speaker'], text=t['text'], source=c['source'], pointer=t['source_pointer']) for c in report['conversations'] for t in c['turns']]
display(pd.DataFrame(examples))
fig, ax = plt.subplots(figsize=(8, 3))
counts = {c['conversation_id']: len(c['turns']) for c in report['conversations']}
ax.bar(list(counts), list(counts.values()), color='#2563eb')
ax.set(ylabel='Authored persona lines', title='Available examples — no reviewed student labels')
plt.tight_layout(); plt.show()

## 3. Worked parser example — synthetic test data

One source file is one conversation. Blank lines and line wraps are not new examples. A speaker prefix begins a turn; continuation lines stay with that speaker. Introductory summaries are retained as a quarantined preamble and excluded from model input, since they can leak hidden facts.

In [ ]:
text = 'Summary: not a student turn.\nStudent: Can you explain more?\n\nPatient: I am unsure.\nI need time.'
c = parse_dialogue(text, 'synthetic notebook mechanics example', 'demo')
display(pd.DataFrame(c['turns']))
print('Quarantined:', c['preamble'])
for turn in c['turns']:
    assert text[turn['start']:turn['end']] == turn['text']

## 4. Annotation schema and exact evidence

Offsets are end-exclusive within the turn. Here `start=0` and `end=len(span)` recover the exact question. Pending annotation is not training truth. A human must approve the action label, with a pseudonymous reviewer ID and timestamp. Rubric mapping is empty for a general dialogue act. A student-skill label requires student speech; an action alone does not establish competent performance or opportunity applicability.

In [ ]:
vocab = json.loads((ROOT / 'backend_kelsch/app/data/action_annotation_vocab.json').read_text())
span = c['turns'][0]['text']
row = dict(annotation_id='demo:0', conversation_id='demo', turn_id='demo:0', speaker='student',
           start=0, end=len(span), span=span, namespace='dialogue_act', labels=['question'],
           rubric_mapping=[], review_status='pending', reviewer_type=None, reviewer_id=None, reviewed_at=None)
display(pd.Series(row))
assert validate_annotations([row], [c], vocab) == []
print('Eligible reviewed examples: 0')

## 5. Leakage prevention

A patient link, script-family link, or identical normalized turn joins conversations into one connected group. Transitive links also stay together. The seeded 60/20/20-style group allocation reserves train, validation and test; tiny datasets can lack training data and must not train. Family and patient IDs need reviewer completion. Unknown IDs cannot guarantee patient independence.

In [ ]:
display(pd.DataFrame([dict(case=k, split=v) for k,v in report['splits'].items()]))
fig, ax = plt.subplots(figsize=(7, 2.5))
statuses = ['Unreviewed persona lines', 'Reviewed action annotations', 'Reviewed student-skill annotations']
ax.barh(statuses, [len(examples), 0, 0], color=['#60a5fa', '#f59e0b', '#10b981'])
ax.set(xlabel='Count', title='Preparation material versus training supervision')
plt.tight_layout(); plt.show()

## 6. Baseline, metrics and LLM comparison

With eligible data, fit word unigram/bigram TF-IDF on train spans only, then one-versus-rest logistic regression. For each label, precision = TP/(TP+FP), recall = TP/(TP+FN), F1 = 2PR/(P+R); macro-F1 gives each label equal weight. Multi-label confusion matrices report TN, FP, FN and TP per label. Inspect support and failures rather than relying on the average.

`evaluate` reports class balance and these metrics once reviewed train/test data exists with positive and negative training examples per label. Separately logged prompted-LLM outputs can be scored on the same test IDs via `predictions`, with model ID and prompt version. No paid inference runs here. No reviewed labels means null metrics, not zero accuracy.

In [ ]:
display(pd.DataFrame(report['evaluation']).T)
result = evaluate([row], [c], vocab, 'dialogue_act')
assert result['status'] == 'NOT EVALUABLE' and result['macro_f1'] is None
print(result['reason'])

## 7. Gates and next implementation

Preparation checks are covered by backend tests: exact offsets, review eligibility, namespace restrictions, unknown-format rejection and transitive group isolation. Proposed model targets are macro-F1 ≥0.85 and reviewer-designated critical-action recall ≥0.95 on held-out groups. Adequate class support and uncertainty review are required; no critical-label set is approved yet.

Notebook 03 can build the controlled simulator using the existing classifier. Trained extraction must remain blocked until reviewed data permits evaluation. Respect the repository's deterministic authored visit branches. This stage provides no evidence of human learning efficiency; later learner experiments will evaluate that separately.

In [ ]:
display(pd.Series(report['gates']))
print(report['advance_to_notebook_03'])
for limitation in report['limitations']:
    print('-', limitation)